# OWOD Replay Protocol V3 — one-click Colab experiment

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gubiczam/owod-active/blob/main/notebooks/owod_active.ipynb)

Open in a fresh **GPU** runtime and choose **Runtime → Run all**. Google Drive
authorization is the only expected interaction. The notebook is pinned to reviewed OWL
and PROB commits, validates the completed no-replay baseline before training, resumes
safe partial runs, and publishes a strict baseline/uniform/tail comparison to Drive.

The scientific protocol is owned by `owl.runner`; these cells only prepare, verify,
orchestrate, audit, and report it. A failed assertion stops the expensive path.


In [ ]:
# 0 — Parameters and immutable experiment identity
# ============================== PARAMETERS ==============================
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

RUN_GPU = True
FAST_CHAIN = True
SELECTION_ARM = "random"
LABELLING_POLICY = "known_plus_selected"
N_TASKS = 6
BUDGET_PER_TASK = 600
ROUNDS_PER_TASK = 6
CANDIDATE_IMAGES = 2000
PROPOSALS_PER_IMAGE = 50
REPLAY_REALLOCATE = False
EPOCHS = 5
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
N_CLUSTERS = 1600
SEED = 0
REPLAY_ARMS = ("uniform", "tail_favouring")
EVAL_MAX_PER_CLASS = 150
EVAL_REMAINDER_RATIO = 1
TIME_BUDGET_MINUTES = 420
SESSION_CEILING_MINUTES = 840

OWL_REPOSITORY = "https://github.com/gubiczam/owod-active.git"
OWL_COMMIT = "ae2d2ab1bdeb7a9c30992448d0a839c3458451e9"
PROB_REPOSITORY = "https://github.com/gubiczam/PROB.git"
PROB_COMMIT = "4c66be1a52cad9360e09c729e9134aba8fe0b531"

DRIVE_ROOT = "/content/drive/MyDrive/OWL"
CHECKPOINT_RELATIVE = "checkpoints/SOWODB/t1.pth"
BASELINE_NAME = "random__none"
PLANNED_RUNS = ("random__uniform", "random__tail_favouring")
COMPARISON_NAME = "replay_v3_fast_seed0"
SESSION_STARTED = time.monotonic()
RUN_STATUS = {}
VALIDATION_OUTPUTS = {}

assert RUN_GPU and FAST_CHAIN
assert (N_TASKS, BUDGET_PER_TASK, ROUNDS_PER_TASK) == (6, 600, 6)
assert (CANDIDATE_IMAGES, PROPOSALS_PER_IMAGE) == (2000, 50)
assert (EPOCHS, LEARNING_RATE, BATCH_SIZE, N_CLUSTERS, SEED) == (5, 2e-4, 2, 1600, 0)
assert REPLAY_ARMS == ("uniform", "tail_favouring") and not REPLAY_REALLOCATE
EXPERIMENT_AUDIT = {
    "selection": SELECTION_ARM, "labelling": LABELLING_POLICY,
    "tasks": N_TASKS, "annotation_budget": BUDGET_PER_TASK,
    "rounds": ROUNDS_PER_TASK, "candidate_images": CANDIDATE_IMAGES,
    "proposals_per_image": PROPOSALS_PER_IMAGE,
    "replay_arms": REPLAY_ARMS, "replay_reallocate": REPLAY_REALLOCATE,
    "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE, "clusters": N_CLUSTERS, "seed": SEED,
    "per_run_minutes": TIME_BUDGET_MINUTES,
    "session_ceiling_minutes": SESSION_CEILING_MINUTES,
    "workspaces": (BASELINE_NAME, *PLANNED_RUNS),
}
print("Pinned experiment:", OWL_COMMIT[:12], PROB_COMMIT[:12])
print(json.dumps(EXPERIMENT_AUDIT, indent=2))


In [ ]:
# 1 — Mount Drive and prove the persistent root is writable
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE = Path(DRIVE_ROOT)
DRIVE.mkdir(parents=True, exist_ok=True)
_drive_probe = DRIVE / ".owod_write_probe"
_drive_probe.write_text("ok", encoding="utf-8")
assert _drive_probe.read_text(encoding="utf-8") == "ok"
_drive_probe.unlink()
print("Drive writable:", DRIVE)


In [ ]:
# 2 — Pin OWL exactly, install its declared dependencies, and import fresh code
def _checked(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)


def _capture(command, **kwargs):
    return _checked(command, capture_output=True, **kwargs).stdout.strip()


def _normalise_git_url(value):
    value = value.strip().removesuffix(".git").rstrip("/")
    if value.startswith("git@github.com:"):
        value = "https://github.com/" + value.split(":", 1)[1]
    return value


def ensure_pinned_checkout(path, repository, commit):
    path = Path(path)
    expected = _normalise_git_url(repository)
    if path.exists():
        assert (path / ".git").is_dir(), f"Refusing non-git path: {path}"
        origin = _normalise_git_url(_capture(["git", "remote", "get-url", "origin"], cwd=path))
        assert origin == expected, f"Refusing unexpected origin at {path}: {origin}"
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        _checked(["git", "clone", "--filter=blob:none", "--no-checkout", repository, str(path)])
    _checked(["git", "fetch", "--depth", "1", "origin", commit], cwd=path)
    _checked(["git", "reset", "--hard", commit], cwd=path)
    _checked(["git", "clean", "-fdx"], cwd=path)
    actual = _capture(["git", "rev-parse", "HEAD"], cwd=path)
    assert actual == commit, f"{path}: expected {commit}, got {actual}"
    return path


ROOT = ensure_pinned_checkout(Path("/content/owod-active"), OWL_REPOSITORY, OWL_COMMIT)
_checked([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q",
          "-e", f"{ROOT}[plots]"])

for _name in [n for n in sys.modules if n == "owl" or n.startswith("owl.")]:
    del sys.modules[_name]
sys.path.insert(0, str(ROOT))

from owl import bridge, comparison, evaluation_subset, exemplars, metrics, protocol, replay, runner

_required = {
    "run_chain.prepare_images": "prepare_images" in __import__("inspect").signature(runner.run_chain).parameters,
    "CycleConfig.replay_protocol_version": "replay_protocol_version" in runner.CycleConfig.__dataclass_fields__,
    "comparison.compatibility": hasattr(comparison, "compatibility"),
    "metrics.validate_per_class_ap50": hasattr(metrics, "validate_per_class_ap50"),
}
assert all(_required.values()), {k: v for k, v in _required.items() if not v}
OWL_SHA = _capture(["git", "rev-parse", "HEAD"], cwd=ROOT)
print("OWL ready:", OWL_SHA, "from", ROOT)


In [ ]:
# 3 — Pin and validate the reviewed PROB bridge; build the optional CUDA kernel
PROB = ensure_pinned_checkout(Path("/content/PROB"), PROB_REPOSITORY, PROB_COMMIT)
_checked([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q",
          "-r", str(PROB / "requirements.txt")])


def msda_available():
    probe = subprocess.run(
        [
            sys.executable,
            "-c",
            (
                "import importlib; "
                "importlib.invalidate_caches(); "
                "import MultiScaleDeformableAttention"
            ),
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    return probe.returncode == 0


if not msda_available():
    _checked([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", "ninja"])
    build = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
         "--no-build-isolation", "."], cwd=PROB / "models" / "ops", text=True,
        capture_output=True,
        check=False,
    )
    if build.returncode != 0:
        print("WARNING: CUDA extension build failed; PROB's correct PyTorch fallback will be slower.")
        print("\n".join((build.stdout + "\n" + build.stderr).splitlines()[-20:]))

MSDA_BUILT = msda_available()
PROB_SHA = _capture(["git", "rev-parse", "HEAD"], cwd=PROB)
assert PROB_SHA == PROB_COMMIT
print("PROB ready:", PROB_SHA, "CUDA extension:", "compiled" if MSDA_BUILT else "fallback")


In [ ]:
# 4 — Prepare the canonical OWOD data root and deterministic shared test split
from concurrent.futures import ThreadPoolExecutor

DATA = Path("/content/data/OWOD")
WORK = DRIVE / "work"
CHECKPOINT = DRIVE / CHECKPOINT_RELATIVE
DATA.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
(DATA / "ImageSets" / "OWDETR").mkdir(parents=True, exist_ok=True)

POOL_ARCHIVE = ROOT / "data" / "staging" / "owdetr_pool_annotations.tar.gz"
TEST_ARCHIVE = ROOT / "data" / "staging" / "owdetr_test_annotations.tar.gz"
REPLAY_ARCHIVE = ROOT / "data" / "staging" / "owdetr_replay_annotations.tar.gz"
for _archive in (POOL_ARCHIVE, TEST_ARCHIVE, REPLAY_ARCHIVE):
    assert _archive.is_file(), f"Missing committed archive: {_archive}"


def extract_committed_archive(source, target):
    target = Path(target).resolve()
    with tarfile.open(source) as handle:
        for member in handle.getmembers():
            destination = (target / member.name).resolve()
            assert destination == target or target in destination.parents, member.name
        try:
            handle.extractall(target, filter="data")
        except TypeError:  # older Python, after the path traversal check above
            handle.extractall(target)


for _archive in (POOL_ARCHIVE, TEST_ARCHIVE, REPLAY_ARCHIVE):
    extract_committed_archive(_archive, DATA)

chain = protocol.build_chain(N_TASKS)
declared = [task.new_class for task in chain[1:]]
subset = evaluation_subset.from_archive(
    TEST_ARCHIVE, declared, seed=SEED,
    remainder_multiplier=EVAL_REMAINDER_RATIO, max_per_class=EVAL_MAX_PER_CLASS,
)
TEST_SET = evaluation_subset.SHARED_TEST_SET
evaluation_subset.write_image_set(
    DATA / "ImageSets" / "OWDETR" / f"{TEST_SET}.txt", subset)

candidate_index = json.loads(
    (ROOT / "data" / "reference" / "per_image_class_counts.json").read_text(encoding="utf-8"))
replay_index = json.loads(
    (ROOT / "data" / "reference" / "t1_replay_class_counts.json").read_text(encoding="utf-8"))
assert candidate_index and replay_index and (DATA / "Annotations").is_dir()

JPEG = DATA / "JPEGImages"
JPEG.mkdir(parents=True, exist_ok=True)


def fetch_images(image_ids, workers=32):
    image_ids = [str(value) for value in image_ids]
    missing = [i for i in image_ids if not (JPEG / f"{i}.jpg").is_file()]

    def fetch(image_id):
        target = JPEG / f"{image_id}.jpg"
        for split in ("train2017", "val2017"):
            subprocess.run(
                ["curl", "-sfL", "--retry", "3", "--retry-delay", "1", "-o", str(target),
                 f"https://images.cocodataset.org/{split}/{image_id}.jpg"],
                check=False,
            )
            if target.is_file() and target.stat().st_size > 0:
                return
        target.unlink(missing_ok=True)

    if missing:
        with ThreadPoolExecutor(max_workers=workers) as pool:
            list(pool.map(fetch, missing))
    return [i for i in image_ids if (JPEG / f"{i}.jpg").is_file()]


print("Data ready:", len(candidate_index), "candidate images,", len(replay_index),
      "replay images,", len(subset.image_ids), "shared-test images")


In [ ]:
# 5 — Build the exact fingerprint and run a concise fail-closed preflight
import csv
import re
from dataclasses import replace

import torch

base_config = runner.CycleConfig(
    n_tasks=N_TASKS, budget_per_task=BUDGET_PER_TASK,
    rounds_per_task=ROUNDS_PER_TASK,
    candidate_images_per_task=CANDIDATE_IMAGES,
    proposals_per_image=PROPOSALS_PER_IMAGE,
    arm=SELECTION_ARM, labelling_policy=LABELLING_POLICY,
    replay_arm="uniform", replay_reallocate=REPLAY_REALLOCATE,
    replay_protocol_version=3, epochs=EPOCHS,
    learning_rate=LEARNING_RATE, batch_size=BATCH_SIZE,
    n_clusters=N_CLUSTERS, seed=SEED, measure_grouped_recall=True,
)


def expected_fingerprint(replay_arm):
    return replace(base_config, replay_arm=replay_arm).fingerprint()


def fingerprint_differences(path, expected):
    stamp = Path(path) / "config.json"
    if not stamp.exists():
        return {}
    stored = json.loads(stamp.read_text(encoding="utf-8"))
    return {name: (stored.get(name, "(absent)"), value)
            for name, value in expected.items() if stored.get(name, "(absent)") != value}


def workspace_problem(path):
    path = Path(path)
    if not path.exists():
        return ""
    completed = []
    for task in [f"t{i}" for i in range(2, N_TASKS + 1)]:
        task_dir = path / f"{task}_{SELECTION_ARM}"
        state, scored = task_dir / "state.json", task_dir / "metrics.json"
        if state.exists() != scored.exists():
            return f"{task_dir} has only one of state.json and metrics.json"
        if state.exists():
            completed.append(task)
    expected_prefix = [f"t{i}" for i in range(2, 2 + len(completed))]
    if completed != expected_prefix:
        return f"non-prefix completed tasks: {completed}"
    results = path / f"results_{SELECTION_ARM}.csv"
    if results.exists():
        with results.open(newline="", encoding="utf-8") as handle:
            recorded = [row["task"] for row in csv.DictReader(handle)]
        if recorded != completed:
            return f"results tasks {recorded} disagree with states {completed}"
    elif completed:
        return f"{completed} have state/metrics but results CSV is missing"
    return ""


BASELINE = WORK / BASELINE_NAME
TARGETS = {arm: WORK / f"random__{arm}" for arm in REPLAY_ARMS}
gpu_probe = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=False,
)
gpu_memory_match = re.search(r"(\d+)\s*MiB", gpu_probe.stdout)
GPU_MEMORY_MIB = int(gpu_memory_match.group(1)) if gpu_memory_match else 0
DRIVE_FREE_GB = shutil.disk_usage(DRIVE).free / 2**30
LOCAL_FREE_GB = shutil.disk_usage(DATA).free / 2**30
package_probe = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
    check=False,
)
prob_bridge = bridge.Bridge(
    prob_root=PROB, data_root=DATA, log_dir=DRIVE / "logs", num_workers=2, seed=SEED)

baseline_diff = fingerprint_differences(BASELINE, expected_fingerprint("none"))
target_diffs = {arm: fingerprint_differences(path, expected_fingerprint(arm))
                for arm, path in TARGETS.items()}
target_problems = {arm: workspace_problem(path) for arm, path in TARGETS.items()}
checks = {
    "GPU visible": gpu_probe.returncode == 0 and bool(gpu_probe.stdout.strip()),
    "GPU memory >= 14 GiB": GPU_MEMORY_MIB >= 14_000,
    "torch CUDA": bool(torch.cuda.is_available()),
    "package consistency": package_probe.returncode == 0,
    "OWL exact SHA": OWL_SHA == OWL_COMMIT,
    "PROB exact SHA": PROB_SHA == PROB_COMMIT,
    "Drive writable": DRIVE.is_dir() and os.access(DRIVE, os.W_OK),
    "Drive free >= 8 GiB": DRIVE_FREE_GB >= 8.0,
    "local free >= 12 GiB": LOCAL_FREE_GB >= 12.0,
    "checkpoint": CHECKPOINT.is_file(),
    "canonical data root": all((DATA / name).is_dir() for name in ("Annotations", "JPEGImages", "ImageSets")),
    "completed baseline exists": BASELINE.is_dir() and (BASELINE / "results_random.csv").is_file(),
    "baseline fingerprint": not baseline_diff,
    "target fingerprints": not any(target_diffs.values()),
    "target artefact integrity": not any(target_problems.values()),
    "Replay Protocol V3": base_config.replay_protocol_version == 3,
    "workspace isolation": len({BASELINE.resolve(), *(p.resolve() for p in TARGETS.values())}) == 3
                           and BASELINE.name not in PLANNED_RUNS,
}
try:
    BRIDGE_CHECK = prob_bridge.check()
    checks["PROB bridge flags"] = True
except Exception as error:  # noqa: BLE001
    BRIDGE_CHECK = {"error": str(error)}
    checks["PROB bridge flags"] = False

print(f"{'preflight':30s} status")
print("-" * 40)
for name, passed in checks.items():
    print(f"{name:30s} {'PASS' if passed else 'FAIL'}")
if baseline_diff:
    print("baseline differences:", baseline_diff)
if any(target_diffs.values()):
    print("target differences:", target_diffs)
if any(target_problems.values()):
    print("target artefact problems:", target_problems)
if package_probe.returncode:
    print("package conflicts:", package_probe.stdout + package_probe.stderr)
failed = [name for name, passed in checks.items() if not passed]
assert not failed, f"PREFLIGHT FAILED: {failed}. No GPU evaluation or training was started."
PREFLIGHT_OK = True
GPU_NAME = torch.cuda.get_device_name(0)
print("PREFLIGHT PASS —", GPU_NAME, "| torch", torch.__version__, "| CUDA", torch.version.cuda)


In [ ]:
# 6 — Verify the historical baseline, validate/create its anchor, compare before training
EXPECTED_TASKS = [task.name for task in chain[1:]]


def validate_detection_run(run, replay_arm, require_anchor=True, allow_partial=False):
    assert run is not None, f"Missing run random__{replay_arm}"
    wanted_tasks = EXPECTED_TASKS[:len(run.tasks)] if allow_partial else EXPECTED_TASKS
    assert run.tasks and run.tasks == wanted_tasks, (run.name, run.tasks, wanted_tasks)
    assert run.config == expected_fingerprint(replay_arm), f"Fingerprint mismatch: {run.name}"
    assert set(run.per_task_ap) == set(wanted_tasks), run.per_task_ap.keys()
    assert all(len(values) == 81 for values in run.per_task_ap.values())
    assert set(run.per_task_recall) == set(wanted_tasks), run.per_task_recall.keys()
    for task in wanted_tasks:
        report = run.per_class_checks.get(task, {})
        assert report.get("usable") and report.get("checks"), (run.name, task, report)
    if require_anchor:
        assert len(run.anchor_ap) == 81, f"{run.name}: missing/invalid 81-class anchor"
        anchor_report = run.per_class_checks.get("anchor", {})
        assert anchor_report.get("usable") and anchor_report.get("checks"), anchor_report
        per_class = comparison.table_per_class({run.name: run})
        assert len(per_class) == len(protocol.TASK1)
        assert all(row.get(f"{run.name}:forgetting") is not None for row in per_class)
    return run


baseline = comparison.load_run(BASELINE)
validate_detection_run(baseline, "none", require_anchor=False)
ready = fetch_images(subset.image_ids)
assert ready == list(subset.image_ids), "Not every shared-test image downloaded"

anchor_command = [
    sys.executable, str(ROOT / "tools" / "evaluate_anchor.py"),
    "--workspace", str(BASELINE), "--checkpoint", str(CHECKPOINT),
    "--prob-root", str(PROB), "--data-root", str(DATA),
    "--archive", str(TEST_ARCHIVE),
    "--eval-max-per-class", str(EVAL_MAX_PER_CLASS),
    "--eval-remainder-ratio", str(EVAL_REMAINDER_RATIO),
]
_checked([*anchor_command, "--dry-run"])
if not (BASELINE / "anchor_metrics.json").is_file():
    _checked(anchor_command)
    RUN_STATUS[BASELINE_NAME] = "anchor generated; baseline reused"
else:
    _checked(anchor_command)  # re-verifies every input, then preserves the existing anchor
    RUN_STATUS[BASELINE_NAME] = "validated and reused"

baseline = validate_detection_run(comparison.load_run(BASELINE), "none", require_anchor=True)
for _arm, _path in TARGETS.items():
    _existing = comparison.load_run(_path)
    if _existing is not None:
        validate_detection_run(_existing, _arm, require_anchor=True, allow_partial=True)
pre_runs = comparison.load_runs(WORK)
pre_clashes = comparison.compatibility(pre_runs, reference=BASELINE_NAME)
assert not pre_clashes, f"Incompatible existing runs before training: {pre_clashes}"
PRECOMPARE = Path("/content/owod_preflight_comparison")
shutil.rmtree(PRECOMPARE, ignore_errors=True)
_checked([sys.executable, str(ROOT / "tools" / "compare_replay.py"), str(WORK),
          "--out", str(PRECOMPARE), "--include", ",".join(comparison.EXPECTED),
          "--no-plots"])
assert (PRECOMPARE / "summary.json").is_file()
VALIDATION_OUTPUTS[BASELINE_NAME] = str(BASELINE / "anchor_metrics.json")
print("Baseline PASS:", EXPECTED_TASKS, "| anchor/per-class/recall/forgetting validated")


In [ ]:
# 7 — Run or resume random__uniform (420-minute per-run cap)
def completed_depth(path):
    found = comparison.load_run(path)
    return len(found) if found is not None else 0


def run_replay_arm(replay_arm):
    target = TARGETS[replay_arm]
    before = completed_depth(target)
    gpu_minutes_before = float(prob_bridge.cost_report()["total"])
    assert gpu_minutes_before < SESSION_CEILING_MINUTES, (
        f"Session ceiling reached before random__{replay_arm}. Reconnect and Run all; the run resumes safely.")
    per_run_budget = min(TIME_BUDGET_MINUTES, SESSION_CEILING_MINUTES - gpu_minutes_before)
    assert 0 < per_run_budget <= TIME_BUDGET_MINUTES
    rows = runner.run_chain(
        prob_bridge, replace(base_config, replay_arm=replay_arm),
        workspace=target, candidate_index=candidate_index, replay_index=replay_index,
        replay_root=DATA, start_checkpoint=CHECKPOINT, test_set=TEST_SET, chain=chain,
        time_budget_minutes=per_run_budget, prepare_images=fetch_images,
    )
    after = len(rows)
    RUN_STATUS[f"random__{replay_arm}"] = (
        "validated and skipped" if before == len(EXPECTED_TASKS)
        else "resumed and completed" if before > 0 and after == len(EXPECTED_TASKS)
        else "completed" if after == len(EXPECTED_TASKS)
        else f"partial ({after}/{len(EXPECTED_TASKS)}); Run all again to resume"
    )
    return rows


by_arm = {}
by_arm["random__uniform"] = run_replay_arm("uniform")
assert len(by_arm["random__uniform"]) == len(EXPECTED_TASKS), RUN_STATUS["random__uniform"]


In [ ]:
# 8 — Strict Replay Protocol V3 audit for random__uniform
def validate_replay_workspace(replay_arm):
    name = f"random__{replay_arm}"
    path = TARGETS[replay_arm]
    run = validate_detection_run(comparison.load_run(path), replay_arm, require_anchor=True)
    budget = int(replay.ARMS[replay_arm]["total"])
    assert budget == 400
    prior_exemplars = set()
    prior_task_images = set()
    task_audits = []
    for index, task in enumerate(EXPECTED_TASKS):
        state_path = path / f"{task}_{SELECTION_ARM}" / "state.json"
        assert state_path.is_file(), state_path
        state = json.loads(state_path.read_text(encoding="utf-8"))
        diagnostics = state["replay_row"]
        assert (diagnostics["requested_objects"] == diagnostics["allocated_objects"]
                == diagnostics["delivered_objects"] == budget), (task, diagnostics)
        current = {tuple(row) for row in state["exemplars"]}
        assert len(current) == budget
        per_class = comparison.parse_per_class_quota(diagnostics["per_class"])
        assert sum(per_class.values()) == budget
        rebuilt = {}
        for _, class_name, _ in current:
            rebuilt[class_name] = rebuilt.get(class_name, 0) + 1
        assert rebuilt == per_class
        sources = {row[0] for row in current}
        assert sources.isdisjoint(state["previous_task_images"]), f"{task}: replayed current-task source"
        assert diagnostics["images"] == diagnostics["unique_source_images"] == len(sources)
        aliases = {exemplars.alias_id(source) for source in sources}
        assert len(aliases) == len(sources)
        assert all(exemplars.source_id(alias) in sources for alias in aliases)
        if index == 0:
            assert sources <= set(replay_index), f"{task}: source outside canonical old-data pool"
        else:
            assert all(item in prior_exemplars or item[0] in prior_task_images for item in current), (
                f"{task}: exemplar resurrected outside E_(k-1) union L_(k-1)")
        digest = hashlib.sha256(json.dumps(sorted(current), separators=(",", ":")).encode()).hexdigest()
        task_audits.append({
            "task": task, "requested": budget, "allocated": budget, "delivered": budget,
            "objects": len(current), "source_images": len(sources),
            "per_class_total": sum(per_class.values()), "identity_sha256": digest,
            "from_previous_memory": diagnostics["from_previous_memory"],
            "added": diagnostics["added"], "evicted": diagnostics["evicted"],
        })
        prior_exemplars = current
        prior_task_images = set(state["previous_task_images"])
    audit = {
        "schema": "owl_replay_v3_audit_v1", "run": name,
        "config": run.config, "tasks": task_audits,
        "per_class_validated": run.per_class_ap_is_validated,
        "recall_crosschecks": sorted(run.per_task_recall),
        "resume_identity": "state-restored object identities are SHA-256 recorded per task",
    }
    output = path / "replay_v3_audit.json"
    pending = output.with_suffix(".json.pending")
    pending.write_text(json.dumps(audit, indent=2), encoding="utf-8")
    pending.replace(output)
    VALIDATION_OUTPUTS[name] = str(output)
    print(name, "PASS — requested = allocated = delivered = 400 at t2–t6")
    return run


uniform_run = validate_replay_workspace("uniform")


In [ ]:
# 9 — Run or resume random__tail_favouring (only after uniform passes)
by_arm["random__tail_favouring"] = run_replay_arm("tail_favouring")
assert len(by_arm["random__tail_favouring"]) == len(EXPECTED_TASKS), RUN_STATUS["random__tail_favouring"]


In [ ]:
# 10 — Strict Replay Protocol V3 audit for random__tail_favouring
tail_run = validate_replay_workspace("tail_favouring")
all_runs = comparison.load_runs(WORK)
assert list(all_runs) == list(comparison.EXPECTED), list(all_runs)
assert not comparison.compatibility(all_runs, reference=BASELINE_NAME)
assert all(run.tasks == EXPECTED_TASKS for run in all_runs.values())
print("All three runs are complete, protocol-identical, and comparison-ready.")


In [ ]:
# 11 — Generate the full comparison locally, validate it, then persist it to Drive
LOCAL_COMPARISON = Path("/content/owod_comparison_replay_v3_fast_seed0")
PERSISTENT_COMPARISON = DRIVE / "comparisons" / COMPARISON_NAME
shutil.rmtree(LOCAL_COMPARISON, ignore_errors=True)
_checked([sys.executable, str(ROOT / "tools" / "compare_replay.py"), str(WORK),
          "--out", str(LOCAL_COMPARISON), "--include", ",".join(comparison.EXPECTED)])

table_stems = (
    "depth", "table1_task_comparison", "table2_delta_vs_baseline",
    "table3_tail_vs_uniform", "table4_per_class", "table5_replay_composition",
    "table6_cost",
)
required_outputs = {"summary.json"}
for stem in table_stems:
    required_outputs.update({f"{stem}.csv", f"{stem}.md", f"{stem}.tex"})
required_outputs.update({
    f"figure_{letter}_{suffix}.{extension}"
    for letter, suffix in (
        ("a", "group_ap"), ("b", "forgetting"), ("c", "new_class_ap"),
        ("d", "forgetting_vs_frequency"), ("e", "replay_allocation"),
        ("f", "forgetting_vs_anchor"),
    ) for extension in ("png", "pdf")
})
missing = sorted(name for name in required_outputs if not (LOCAL_COMPARISON / name).is_file())
assert not missing, f"Comparison output incomplete: {missing}"
summary = json.loads((LOCAL_COMPARISON / "summary.json").read_text(encoding="utf-8"))
assert set(summary["runs"]) == set(comparison.EXPECTED)
assert not summary["missing"] and not summary["compatibility_clashes"]

PERSISTENT_COMPARISON.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_COMPARISON, PERSISTENT_COMPARISON, dirs_exist_ok=True)
missing_persistent = sorted(
    name for name in required_outputs if not (PERSISTENT_COMPARISON / name).is_file())
assert not missing_persistent, missing_persistent
VALIDATION_OUTPUTS["comparison"] = str(PERSISTENT_COMPARISON)
print("Comparison PASS:", len(required_outputs), "validated files copied to", PERSISTENT_COMPARISON)


In [ ]:
# 12 — Final audit summary; success marker is printed only after every assertion passed
EXPERIMENT_COMPLETE = True
print("=" * 78)
print("FINAL OWOD REPLAY V3 SUMMARY")
print("=" * 78)
print("OWL SHA: ", OWL_SHA)
print("PROB SHA:", PROB_SHA)
print("GPU:     ", GPU_NAME)
print("torch:   ", torch.__version__, "| CUDA:", torch.version.cuda,
      "| MSDA:", "compiled" if MSDA_BUILT else "PyTorch fallback")
print("runs:")
for name in comparison.EXPECTED:
    print(f"  {name:25s} {RUN_STATUS[name]}")
print("validations / outputs:")
for name, output in VALIDATION_OUTPUTS.items():
    print(f"  {name:25s} {output}")
print("session elapsed minutes:", round((time.monotonic() - SESSION_STARTED) / 60.0, 1),
      "/", SESSION_CEILING_MINUTES)
print("metered GPU minutes:", round(float(prob_bridge.cost_report()["total"]), 1),
      "/", SESSION_CEILING_MINUTES)
print("EXPERIMENT COMPLETE")
